# MemoryVerse: Family 3 MS-COCO Full BLIP-2 & Florence-2 Inference Reproduction

This notebook provides a 100% reproducible GPU pipeline for generating genuine captions using:
1. `Salesforce/blip2-opt-2.7b`
2. `microsoft/Florence-2-large`

on the exact $N=1,000$ MS-COCO Karpathy test cohort used in MemoryVerse Family 3 evaluations.
It indexes the generated captions with `BAAI/bge-base-en-v1.5` and evaluates zero-shot cross-modal retrieval against held-out human query captions.

In [ ]:
!pip install -q torch torchvision transformers timm flash-attn einops sentencepiece accelerate rank_bm25

In [ ]:
import os, json, time
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from transformers import AutoProcessor, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load Karpathy test split
karpathy_path = "coco_karpathy_test.json"
if not os.path.exists(karpathy_path):
    raise FileNotFoundError("Please place coco_karpathy_test.json in working directory.")

with open(karpathy_path) as f:
    data = json.load(f)

rng = np.random.default_rng(42)
selected_idx = rng.choice(len(data), size=1000, replace=False)
eval_data = [data[i] for i in selected_idx]
print(f"Selected {len(eval_data)} images for evaluation.")

In [ ]:
# Run BLIP-2 opt-2.7b caption generation
print("Loading Salesforce/blip2-opt-2.7b...")
blip2_proc = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16, device_map="auto"
)

blip2_captions = []
for item in tqdm(eval_data, desc="BLIP-2 inference"):
    # Load image from local path or URL
    img_path = item.get("filepath", "") + "/" + item.get("filename", "")
    raw_image = Image.open(img_path).convert("RGB")
    inputs = blip2_proc(raw_image, return_tensors="pt").to(device, torch.float16)
    out = blip2_model.generate(**inputs, max_new_tokens=50)
    caption = blip2_proc.decode(out[0], skip_special_tokens=True).strip()
    blip2_captions.append(caption)

with open("blip2_opt2.7b_generated_captions.json", "w") as f:
    json.dump(blip2_captions, f, indent=2)
print("BLIP-2 captions generated and saved.")

In [ ]:
# Run Florence-2 Detailed Caption Generation
print("Loading microsoft/Florence-2-large...")
florence_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Florence-2-large", torch_dtype=torch.float16, trust_remote_code=True
).to(device)
florence_proc = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

florence_captions = []
prompt = "<MORE_DETAILED_CAPTION>"
for item in tqdm(eval_data, desc="Florence-2 inference"):
    img_path = item.get("filepath", "") + "/" + item.get("filename", "")
    raw_image = Image.open(img_path).convert("RGB")
    inputs = florence_proc(text=prompt, images=raw_image, return_tensors="pt").to(device, torch.float16)
    generated_ids = florence_model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )
    generated_text = florence_proc.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = florence_proc.post_process_generation(generated_text, task=prompt, image_size=(raw_image.width, raw_image.height))
    florence_captions.append(parsed_answer[prompt].strip())

with open("florence2_large_detailed_captions.json", "w") as f:
    json.dump(florence_captions, f, indent=2)
print("Florence-2 captions generated and saved.")

In [ ]:
# Retrieval Evaluation with BGE-base
from transformers import AutoTokenizer, AutoModel

print("Evaluating zero-shot retrieval with BGE-base...")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
model = AutoModel.from_pretrained("BAAI/bge-base-en-v1.5").to(device)

def encode_texts(texts):
    embs = []
    for i in range(0, len(texts), 64):
        batch = texts[i:i+64]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**inputs)
            cls_emb = out.last_hidden_state[:, 0, :]
            cls_emb = torch.nn.functional.normalize(cls_emb, p=2, dim=-1)
            embs.append(cls_emb.cpu().numpy())
    return np.vstack(embs)

queries = [item["query"] for item in eval_data]
q_embs = encode_texts(queries)

for name, captions in [("BLIP-2 opt-2.7b", blip2_captions), ("Florence-2 Large", florence_captions)]:
    d_embs = encode_texts(captions)
    sims = np.dot(q_embs, d_embs.T)
    mrr_list = []
    h1, h5, h10 = 0, 0, 0
    for i in range(len(eval_data)):
        ranked = np.argsort(-sims[i])
        rank = np.where(ranked == i)[0][0] + 1
        mrr_list.append(1.0 / rank)
        if rank == 1: h1 += 1
        if rank <= 5: h5 += 1
        if rank <= 10: h10 += 1
    print(f"[{name}] MRR={np.mean(mrr_list):.4f}, Hit@1={h1/len(eval_data):.4f}, Hit@5={h5/len(eval_data):.4f}, Hit@10={h10/len(eval_data):.4f}")